In [ ]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------
# 1. Device selection (ROCm or CPU)
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# Data preparation

In [ ]:
# -------------------------------------------------
# 2. Load & preprocess data
# -------------------------------------------------
iris = load_iris()
X = iris.data
y = iris.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_gpu = scaler.fit_transform(X_train)
X_test_gpu = scaler.transform(X_test)

# Convert to tensors
X_train_gpu = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_train_gpu = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)


# GNG calculation

In [ ]:
import gng_py
import time
config_file = "config_iris.json"
ctx = gng_py.PyContext()

ctx = gng_py.PyContext()
ctx.create_system()
ctx.load_config(config_file)
ctx.init_dataset_vec(X_train.flatten())
start = time.time()
ctx.fit()
end = time.time()
model_string = ctx.get_model_string()
print("Time for gng calculation: ",end-start)

# Result processing

In [ ]:
import json
import numpy as np
import pandas as pd
data = json.loads(model_string)
points = None
edges = None
edge_positions = None
rows = []
for neuron in data["model"]["neurons"]:
    position = neuron["position"]
    id = neuron['id']
    x = position[0]
    y = position[1]
    rows.append({"id": id, "x": x, "y": y})

points = pd.DataFrame(rows)

num_classes = len(np.unique(y_train))


for a in data["model"]["neurons"]:
    a["hits"] = np.array(np.zeros(num_classes))

In [ ]:
df = pd.DataFrame(data["model"]["neurons"])

In [ ]:
import math
X = X_train
y = y_train

num_samples = len(X)
num_weights = len(df.at[0,"position"])
num_neurons = len(df)
input_width = X.shape[1]


# For all samples
for s in range(0,num_samples):
    # get sample position
    sample_pos = X[s]
    # init neuron dist arr
    dist = np.zeros(num_neurons)

    # For all Neurons
    for i,row in df.iterrows():
        # get neuron position
        neuron_pos = row["position"]
        # init distance val
        dist_val = 0.0

        # For all neuron weights (all dimensions)
        for a in range(0,num_classes):
            # add squared distances
            diff = sample_pos[a]-neuron_pos[a]
            dist_val += diff * diff


        # neuron dist val arr[curr neuron] = sqrt of dist val 
        dist[i] = np.sqrt(dist_val)

    # find best neuron (winner)
    best_neuron_idx = np.argmin(dist)
    sample_class = y[s]
  #  curr_id = df.loc[df["id"] == 57]

    hits = df.at[best_neuron_idx,"hits"]
    hits[sample_class] +=1



In [ ]:
sum = 0
classes = [0,0,0]
for _,a in df.iterrows():
    if max(a["hits"]) != 0.0:
        sum +=1
        #print(a["hits"])

print(sum)

In [ ]:
df

# Find BMU's

In [ ]:
# -------------------------------------------------
# Inference: Find best matching neurons for X_test
# -------------------------------------------------
num_samples_test = len(X_test)
num_neurons = len(df)
num_classes = len(np.unique(y_train))
input_width = X_test.shape[1]

# Store results: list of tuples (sample_idx, neuron_idx, distance, neuron_id)
best_neurons = []

# For all test samples
for s in range(num_samples_test):
    # get sample position
    sample_pos = X_test[s]
    # init neuron dist arr
    dist = np.zeros(num_neurons)

    # For all Neurons
    for i, row in df.iterrows():
        # get neuron position
        neuron_pos = row["position"]
       # print(neuron_pos)
        # init distance val
        dist_val = 0.0

        # For all neuron weights (all dimensions)
        for a in range(input_width):
            # add squared distances
            diff = sample_pos[a] - neuron_pos[a]
            dist_val += diff * diff

        # neuron dist val arr[curr neuron] = sqrt of dist val
        dist[i] = np.sqrt(dist_val)

    # Sort neurons by distance (smallest fi

#    print(dist)
    sorted_indices = np.argsort(dist)
#    print(sorted_indices)
    
    # Store all neurons for this sample, sorted by distance
    for rank, neuron_idx in enumerate(sorted_indices):
        neuron_id = df.at[neuron_idx, "id"]
        distance = dist[neuron_idx]
        best_neurons.append({
            "sample_idx": s,
            "rank": rank,
            "neuron_id": neuron_id,
            "distance": distance
        })

# Convert to DataFrame for easy viewing
best_neurons_df = pd.DataFrame(best_neurons)
print(best_neurons_df.head(2))

In [ ]:
best_neurons_df.loc[best_neurons_df["sample_idx"]==19].head(3)

In [ ]:
y_test[19] ,df.loc[df["id"]==66].hits

# Detection

In [ ]:
# -------------------------------------------------
# Assign class labels to samples using top-k neurons
# -------------------------------------------------
num_samples_test = len(X_test)
k = 3  # Use top 3 nearest neurons
sample_predictions = []

for s in range(num_samples_test):
    # Get top k neurons for this sample
    top_k_neurons = best_neurons_df[best_neurons_df["sample_idx"] == s].head(k)
    
    # Initialize class probabilities
    class_probs = np.zeros(num_classes)
    
    # For each of the top k neurons
    for idx, row in top_k_neurons.iterrows():
        neuron_id = row["neuron_id"]
        rank = row["rank"]
        
        # Get neuron hits
        neuron_row = df[df["id"] == neuron_id]
        hits = np.array(neuron_row.iloc[0]["hits"])
        total_hits = np.sum(hits)
        
        # Calculate weight based on rank (lower rank = higher weight)
        weight = 1.0 / (rank + 1)
        
        # Add weighted class probabilities
        if total_hits > 0:
            class_probs += (hits / total_hits) * weight
        else:
            # If no hits, distribute uniformly
            class_probs += (np.ones(num_classes) / num_classes) * weight
    
    # Normalize probabilities
    class_probs = class_probs / np.sum(class_probs)
    
    # Predict class with highest probability
    predicted_class = np.argmax(class_probs)
    
    sample_predictions.append({
        "sample_idx": s,
        "predicted_class": predicted_class,
        "class_probs": class_probs,
        "true_class": y_test[s]
    })

predictions_df = pd.DataFrame(sample_predictions)
print(predictions_df.head(10))

# Calculate accuracy
accuracy = np.mean(predictions_df["predicted_class"] == predictions_df["true_class"])
print(f"\nAccuracy using GNG: {accuracy:.4f}")